<h4> Zadanie1. Zbuduj (zgodnie z tutorialem) model do tłumaczenia tekstu z języka francuskiego na angielski. 
    
- Dodatkowo, podziel na początku zbiór danych na treningowy, testowy, walidacyjny (zaproponuj w jakich proporcjach)
- Zastosuj metrykę BLEU do tego zadania jako miarę oceniającą wygenerowane tłumaczenia
- Potestuj kilka architektur sieci neuronowych, opcjonalnie możesz wykorzystać również GloVe
- Opracuj w formie krótkiego sprawozdania (max 2 strony) najważniejsze kroki analizy oraz zamieść odpowiednie wykresy - spadek funkcji kosztu w kolejnych epokach (na zbiorze treningowym i walidacyjnym), wizualizacje atencji na przykładowych zdaniach, miara BLEU na zbiorze treningowym, walidacyjnym i testowym. Ile danych rozważano?
    
    
https://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html

In [193]:
# Przygotowanie środowiska
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

# Wybór urządzenia do obliczeń: jeśli dostępna jest karta graficzna (GPU), to używa cuda; inaczej CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [194]:
#Definicja kalsy Lang -> potrzebna do budowania słownika
#zdania w indeksy, freq słów
#potrzbne do sieci neutonowej (modelach seq2seq)

SOS_token = 0 #start of sentence
EOS_token = 1 #end of sentence

class Lang:
    def __init__(self, name):
        self.name = name #nazwa języka - name
        self.word2index = {} #słownik słowo:index
        self.word2count = {} #słownik słowo:freq
        self.index2word = {0: "SOS", 1: "EOS"} #słownik index:słowo
        self.n_words = 2  #liczba słów w słowniku (na poczatku tylko 2 tokeny specjakne)

    def addSentence(self, sentence): #dodaj zdanie do słownika słowo po słowie
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word): #dodaj słowo do słownika
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [195]:
#czyszczenie i normalizacja tekstu

# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s) #normalizuje do formy NFD (np ó = o + ́')
        if unicodedata.category(c) != 'Mn' #pomija ogonki, akcenty
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip()) #unicode na ascii
    s = re.sub(r"([.!?])", r" \1", s) #spacja przez znakami
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s) #dziwne znaki na spacje
    return s.strip()

In [196]:
# normalizacja zdań, podział na pary (słowo-tłumaczenia)
# tworzy obiekty Lang dla języka wejściowego i wyjściowego

def readLangs(lang1, lang2, reverse=False):
    print("Reading lines...")

    # Read the file and split into lines
    lines = open('data/%s-%s.txt' % (lang1, lang2), encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs, make Lang instances
    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [197]:
#filtrowanie par -chcemy łatwe pary do trenowania
MAX_LENGTH = 10 #max liczba słów

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
) 

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes) #tylko pary zaczynające sie od eng_prefixes


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [198]:
#przygotowanie danych do trenowania

#odczyt, filtracja, tokenizacja, podział na zbiery
def prepareData(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = readLangs(lang1, lang2, reverse)
    #print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs) #filtracja
    #print("Trimmed to %s sentence pairs" % len(pairs))
    #print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    #print("Counted words:")
    print(input_lang.name, input_lang.n_words,output_lang.name, output_lang.n_words)
    
    print('Data split - 7 : 1.5 : 1.5 ratio')
    #ratio 7 : 1.5 : 1.5
    train_pairs, temp_pairs = train_test_split(pairs, test_size=0.3, random_state=42)
    val_pairs, test_pairs = train_test_split(temp_pairs, test_size=0.5, random_state=42)

    return input_lang, output_lang, train_pairs, test_pairs, val_pairs

input_lang, output_lang, train_pairs, val_pairs, test_pairs = prepareData('eng', 'fra', True)
print(random.choice(train_pairs))

Reading lines...
fra 4601 eng 2991
Data split - 7 : 1.5 : 1.5 ratio
['je suis assez en colere', 'i m pretty angry']


In [ ]:
#enkoder do modelu sekwencyjnego 
# warstwa osadzeń - embedding
# warstwa gru (Gated Recurrent Unit) - zakodowanie sekwencji na wektor
# dodano Lyer Norm w celu poprawienia zbierzności:
# - model szybciej osiąga dobre wyniki podczas uczenia
# - stabilizuje propagację gradientów (mniejsze ryzyko eksplodujących/zanikających gradientów)
# - zmniejsza wrażliwość na inicjalizację wag i rozmiar batcha
# - można zwiekszyc learning rate wtedy
# - troche jak normalzacja batcha ale nie zalezy od batch size
#embedding → LayerNorm → Dropout → GRU

class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__() #klasa bazowa
        self.hidden_size = hidden_size #rozmiar warstwy ukrytej jako atrybut klasy
        self.layer_norm = nn.LayerNorm(hidden_size)
        self.embedding = nn.Embedding(input_size, hidden_size) #indesy słow na wektor o rozmiarze hidden_size
        self.dropout = nn.Dropout(dropout_p)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True) #przetwarza sekwencje wekotrów osadzeń
        

    def forward(self, input):
        embedded = self.embedding(input)          
        embedded = self.layer_norm(embedded)        
        embedded = self.dropout(embedded)  
        output, hidden = self.gru(embedded)
        return output, hidden

In [200]:
#dekoder z mechanizmem uwagi Bahdanau (attention)
#generuje kolejne tokeny sekwencji wyjściowej na podstawie encodera i wcześniejszych stanów
# attention skupa uwage na istotnych częsciahc sekwencji wejściowej
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.3): #zwiększenie dropout by uniknąc przuczenia
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.gru = nn.GRU(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        #pętla generująca kolejne tokeny określonej długości
        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2)
        context, attn_weights = self.attention(query, encoder_outputs)
        input_gru = torch.cat((embedded, context), dim=2)

        output, hidden = self.gru(input_gru, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

In [201]:
#wczytanie i przygotowanie danycg

def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, train_pairs, val_pairs, test_pairs = prepareData('eng', 'fra', True)

    n = len(train_pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(train_pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
   
    for idx, (inp, tgt) in enumerate(val_pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    val_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    val_sampler = RandomSampler(val_data)
    val_dataloader = DataLoader(val_data, sampler=val_sampler, batch_size=batch_size)
    
    for idx, (inp, tgt) in enumerate(test_pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    test_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    test_sampler = RandomSampler(test_data)
    test_dataloader = DataLoader(test_data, sampler=test_sampler, batch_size=batch_size)
    
    return input_lang, output_lang, train_dataloader, val_dataloader, test_dataloader
   

In [202]:
#funckja do jednej epoki treningowej

def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [203]:
#epoka validacyjna
#brak zero_grad() loss.backward() optimizer.step()
#niepotrzebne podczas walidacji:
#- nie aktualizujemy wag modelu w trakcie walidacji 
#- backward() i optimizer.step() służą do nauki 
#- zero_grad() bo nie liczymy gradientów

def val_epoch(dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion):
    encoder.eval()
    decoder.eval()

    total_loss = 0

    with torch.no_grad():
        for data in dataloader:
            input_tensor, target_tensor = data

            encoder_outputs, encoder_hidden = encoder(input_tensor)
            decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

            loss = criterion(
                decoder_outputs.view(-1, decoder_outputs.size(-1)),
                target_tensor.view(-1)
            )

            total_loss += loss.item()

    return total_loss / len(dataloader)


In [204]:
import time
import math

def asMinutes(s): #sekundy na minuty i sekundy
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [205]:
#główna pętla treningowa dla modelu seq2seq z atencją

def evaluate_bleu(dataloader, encoder, decoder):
    smoothie = SmoothingFunction().method4 #funkcja wygładzająca, by bleu nie wynosiło zero przy krótkich zdaniach
    total_score = 0
    n_sentences = 0

    for input_tensor, target_tensor in dataloader:
        with torch.no_grad():
            encoder_outputs, encoder_hidden = encoder(input_tensor) #koduje zdanie wejsciowe
            decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden) #generuje tłumaczenie

        decoded_indices = decoder_outputs.argmax(-1).cpu().numpy() #wybranie najbaardziej prawdopodobnego tokena na kazdym kroku
        target_indices = target_tensor.cpu().numpy() #przenosi dane z GPU na CPU i konwertuje na numy dla łątwiejszego przetwarzania

        for pred, tgt in zip(decoded_indices, target_indices): #for predykcja, zdanie
            pred_seq = [output_lang.index2word[idx] for idx in pred if idx != EOS_token]  #indeksy na słowa i ucnane sekwencji po EOS
            tgt_seq = [output_lang.index2word[idx] for idx in tgt if idx != EOS_token] 

            if len(tgt_seq) > 0: #bleu dla jednej pary
                total_score += sentence_bleu([tgt_seq], pred_seq, smoothing_function=smoothie)
                n_sentences += 1

    return total_score / n_sentences if n_sentences > 0 else 0


import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker

def showPlot(metrics_dict):
    plt.figure(figsize=(10,6))
    for label, points in metrics_dict.items():
        plt.plot(points, label=label)
    plt.xlabel('Epochs')
    plt.ylabel('Value')
    plt.legend()
    plt.grid(True)
    plt.show()

def train(train_dataloader, val_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = [] #lista strat
    train_bleus, val_bleus = [], []
    train_losses, val_losses = [], []

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        #trening
        train_loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        train_bleu = evaluate_bleu(train_dataloader, encoder, decoder)
        
        #walidacja
        val_loss = val_epoch(val_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        val_bleu = evaluate_bleu(val_dataloader, encoder, decoder)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_bleus.append(train_bleu)
        val_bleus.append(val_bleu)

        if epoch % print_every == 0:
            print(f"{timeSince(start, epoch / n_epochs)} Epoch {epoch}/{n_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train BLEU: {train_bleu:.4f} | Val BLEU: {val_bleu:.4f}")

    showPlot({
        'Train Loss': train_losses,
        'Val Loss': val_losses,
        'Train BLEU': train_bleus,
        'Val BLEU': val_bleus
    })

In [206]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [207]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(test_pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [208]:
hidden_size = 128
batch_size = 32

input_lang, output_lang, train_dataloader, val_dataloader, test_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, val_dataloader, encoder, decoder, 30, print_every=5, plot_every=5)

Reading lines...
fra 4601 eng 2991
Data split - 7 : 1.5 : 1.5 ratio
3m 19s (- 16m 38s) Epoch 5/30 | Train Loss: 1.1312 | Val Loss: 1.2044 | Train BLEU: 0.4022 | Val BLEU: 0.3749
7m 4s (- 14m 8s) Epoch 10/30 | Train Loss: 0.5283 | Val Loss: 0.7368 | Train BLEU: 0.6523 | Val BLEU: 0.5912
10m 41s (- 10m 41s) Epoch 15/30 | Train Loss: 0.2189 | Val Loss: 0.5374 | Train BLEU: 0.8725 | Val BLEU: 0.7730
13m 59s (- 6m 59s) Epoch 20/30 | Train Loss: 0.0933 | Val Loss: 0.4780 | Train BLEU: 0.9464 | Val BLEU: 0.8355
18m 0s (- 3m 36s) Epoch 25/30 | Train Loss: 0.0492 | Val Loss: 0.4764 | Train BLEU: 0.9675 | Val BLEU: 0.8525
20m 46s (- 0m 0s) Epoch 30/30 | Train Loss: 0.0395 | Val Loss: 0.4917 | Train BLEU: 0.9693 | Val BLEU: 0.8543


C:\Users\kajaw\AppData\Local\Temp\ipykernel_12336\3368938443.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [212]:
print(evaluate_bleu(test_dataloader, encoder, decoder))

0.8416830753343091


In [213]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> je suis vraiment desole de t avoir derange
= i m very sorry to have troubled you
< i m very sorry to have disturbed you <EOS>

> il est photogenique
= he s photogenic
< he is photogenic <EOS>

> je ne suis personne de special
= i m no one special
< i m no hurry <EOS>

> je fais attention a ne pas trop depenser
= i m careful not to spend too much
< i m not very sorry about it <EOS>

> il n est plus tout jeune
= he s not young anymore
< he is not so young <EOS>

> je n abandonne pas si facilement
= i m no quitter
< i m not giving that you <EOS>

> je n en ai pas termine
= i m not finished
< i m not done done <EOS>

> il n est pas du tout honnete
= he is not honest at all
< he is not in good physical <EOS>

> je comprends
= i m catching on
< i m sleepy <EOS>

> elles sont en train de se disputer
= they re arguing
< they re taking wrong <EOS>



In [218]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import os

def showAttention(input_sentence, output_words, attentions, save_path='attention_plots'):
    # Utwórz folder, jeśli nie istnieje
    os.makedirs(save_path, exist_ok=True)

    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(attentions.cpu().detach().numpy(), cmap='bone')
    fig.colorbar(cax)

    # Setup axes
    input_words = input_sentence.split(' ') + ['<EOS>']
    ax.set_xticks(range(len(input_words)))
    ax.set_xticklabels(input_words, rotation=90)

    ax.set_yticks(range(len(output_words)))
    ax.set_yticklabels(output_words)

    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.tight_layout()

    # Zapisz wykres
    filename = "_".join(input_sentence.split())[:30]  # bezpieczna nazwa pliku
    filepath = os.path.join(save_path, f"{filename}.png")
    plt.savefig(filepath)
    plt.close()  # zamknij wykres, by nie zjadać pamięci

def evaluateAndShowAttention(input_sentence, save_path='attention_plots'):
    output_words, attentions = evaluate(encoder, decoder, input_sentence, input_lang, output_lang)
    print('input =', input_sentence)
    print('output =', ' '.join(output_words))
    showAttention(input_sentence, output_words, attentions[0, :len(output_words), :], save_path=save_path)

evaluateAndShowAttention('il n est pas aussi grand que son pere')

evaluateAndShowAttention('je suis trop fatigue pour conduire')

evaluateAndShowAttention('je suis desole si c est une question idiote')

evaluateAndShowAttention('je suis reellement fiere de vous')

input = il n est pas aussi grand que son pere
output = he is not as tall as his father <EOS>
input = je suis trop fatigue pour conduire
output = i m too tired to drive <EOS>
input = je suis desole si c est une question idiote
output = i m sorry if this is a stupid question <EOS>
input = je suis reellement fiere de vous
output = i m really proud of you <EOS>
